# Predict which customers may leave with Python

**Your first AutoML classification model: customer churn**

Imagine you work for a phone company. You have a spreadsheet of how customers
used the service, and you want to identify customers who might leave. Could
previous usage help you decide which customers deserve a closer look?

We will train a model to estimate **churn**, meaning a customer leaving the
service. Predicting a category is called *classification*. Our two categories
are stayed and churned. We will also predict a probability, then check how often
the model misses customers who leave or flags customers who stay.

You will save a model, read its report, and reuse it for another CSV. The example
teaches prediction; it does not show that contacting someone will prevent churn.

## First, look at the customers

We use [UCI's Iranian Churn dataset](https://doi.org/10.24432/C5JW3Z). It contains
3,150 customer records. These are five real rows, showing a few of the columns:

| Subscription (months) | Call seconds | Text messages | Tariff | Churn |
| ---: | ---: | ---: | --- | ---: |
| 38 | 4370 | 5 | Pay as you go | 0 |
| 39 | 318 | 7 | Pay as you go | 0 |
| 37 | 2453 | 359 | Pay as you go | 0 |
| 38 | 4198 | 1 | Pay as you go | 0 |
| 38 | 2393 | 2 | Pay as you go | 0 |

The first customer had subscribed for 38 months and did not churn. `Churn` is
our **target**: 1 means churned and 0 means stayed. The other selected columns
are **inputs**, also called features. Never include the target among the inputs
when asking the model for a new prediction.

Only **495 of 3,150 rows (15.7%)** have churn equal to 1. Predicting “stayed” for
everyone would therefore look accurate while missing every customer who leaves.
That is why we will look beyond accuracy.

The publisher describes inputs from the first nine months and outcomes at month
twelve. There are no dates or customer IDs in the downloaded file. Our split
checks held-out profiles from this historical sample, not performance on a later
period. See the [dataset review](https://supervised.mljar.com/assets/customer-churn/dataset-review/).


## Set up your notebook

Use Python 3.11 or 3.12. Install into the notebook's environment before launching it:

```bash
python -m pip install 'mljar-supervised==1.4.0b1' jupyterlab ipywidgets
python -m jupyterlab
```

Run cells from top to bottom. This notebook includes all code; it does not need a
helper script. Standard CPU algorithms use a 120-second training budget and two
threads; total time also includes imports, reports, and reload. There is no GPU
requirement. Package installation is larger than the 132 kB dataset download.
Use a new output directory each time you train. If a cell fails, fix the cause,
restart the kernel, choose a new directory, and run from the top.


## 1. Inspect customers and choose inputs

## Choose information available before churn

The ten inputs describe call failures, complaints, subscription length, charge
band, call seconds and counts, text messages, distinct called numbers, tariff,
and age. Tariff is converted from numeric codes to meaningful category names.
The charge band is an ordinal code, not a money amount.

We omit account `Status`, which may be too close to the outcome for the intended
prediction point; `Customer Value`, whose formula is not documented; and the
redundant `Age Group`. These are deliberate teaching choices. They do not prove
that the original columns leak future information. Review the actual availability
of every input when adapting the example to your data.


In [ ]:
import argparse
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys
import textwrap
import time
from pathlib import Path
from urllib.request import urlopen

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    brier_score_loss, confusion_matrix, log_loss, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from threadpoolctl import threadpool_limits
from supervised import AutoML

DATA_URL = "https://archive.ics.uci.edu/static/public/563/data.csv"
DATA_SHA256 = "90d5fb6bd1630cd4de4b4d28fcf8b4cb92a8f6ab7484605b0799d47386f7dbe1"
FEATURES = [
    "Call Failure", "Complains", "Subscription Length", "Charge Amount",
    "Seconds of Use", "Frequency of use", "Frequency of SMS",
    "Distinct Called Numbers", "Tariff Plan", "Age",
]
TARGET = "Churn"
SEED = 42
THRESHOLD = 0.5  # Chosen before evaluation; not tuned on the test set.


def load_data(path):
    """Download the reviewed snapshot once; retain the original CSV unchanged."""
    if not path.exists():
        content = urlopen(DATA_URL, timeout=60).read()
        if hashlib.sha256(content).hexdigest() != DATA_SHA256:
            raise ValueError("UCI snapshot changed; review the dataset before continuing.")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != DATA_SHA256:
        raise ValueError("Dataset checksum differs from the reviewed snapshot.")
    data = pd.read_csv(path)
    data.columns = [" ".join(name.split()) for name in data.columns]
    expected = set(FEATURES + [TARGET, "Status", "Customer Value", "Age Group"])
    if data.shape != (3150, 14) or set(data.columns) != expected:
        raise ValueError("Unexpected churn dataset schema.")
    if not np.isfinite(data.to_numpy(dtype=float)).all():
        raise ValueError("The reviewed snapshot should contain only finite numbers.")
    if set(data[TARGET]) != {0, 1}:
        raise ValueError("Expected 0 (no churn) and 1 (churn).")
    return data


def evaluate(y, probabilities):
    """Use the same metrics and fixed threshold for baseline and AutoML."""
    labels = (probabilities >= THRESHOLD).astype(int)
    return {
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
        "log_loss": log_loss(y, probabilities, labels=[0, 1]),
        "brier_score": brier_score_loss(y, probabilities),
        "accuracy": accuracy_score(y, labels),
        "precision": precision_score(y, labels, zero_division=0),
        "recall": recall_score(y, labels, zero_division=0),
        "confusion_matrix": confusion_matrix(y, labels, labels=[0, 1]).tolist(),
    }



from IPython.display import Image, display


In [ ]:
output = Path("AutoML_CustomerChurn_Notebook").resolve()
data_path = Path("AutoML_ChurnData/customer-churn.csv").resolve()
budget = 120
thread_limits = threadpool_limits(limits=2)
# Step 1. Inspect the customers and choose the target.
started = time.perf_counter()
output.mkdir(parents=True, exist_ok=False)
data = load_data(data_path)
print(f"Customers: {len(data)}; churned: {data[TARGET].sum()} ({data[TARGET].mean():.1%}).", flush=True)
print(data.head().to_string(index=False), flush=True)
# Omit Status (proximity to churn), undocumented Customer Value, and redundant Age Group.
X, y = data[FEATURES].copy(), data[TARGET]
# Tariff codes name plans, not a numeric scale. AutoML handles their encoding.
X["Tariff Plan"] = X["Tariff Plan"].map({1: "Pay as you go", 2: "Contractual"})
display(X.head())


## 2. Reserve test profiles

## Keep repeated profiles together

This file contains repeated records. We group identical combinations of the ten
inputs so the same profile cannot occur in both training and testing. We keep
all rows, including profiles with differing outcomes; grouping never uses the
target. Without IDs, we cannot establish whether matching rows are the same person.

The fixed split uses 2,511 rows for training and 639 for testing. Within training,
three grouped folds supply the validation data used to choose models. The script
checks group separation and saves `splits.json`. Grouping and stratification
approximately preserve the churn proportion; they do not create a time split.


In [ ]:
# Step 2. Reserve test customers and keep identical input profiles together.
# There are no customer IDs or dates. These groups are only duplicate-profile proxies.
groups = X.groupby(FEATURES, sort=True, dropna=False).ngroup()
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train, test = next(outer.split(X, y, groups))
X_train, y_train = X.iloc[train].reset_index(drop=True), y.iloc[train].reset_index(drop=True)
X_test, y_test = X.iloc[test], y.iloc[test]
train_groups = groups.iloc[train].reset_index(drop=True)
cv = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(
    X_train, y_train, train_groups))
assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
for fit_rows, valid_rows in cv:
    assert set(train_groups.iloc[fit_rows]).isdisjoint(train_groups.iloc[valid_rows])
split_record = {
    "seed": SEED, "train_rows": train.tolist(), "test_rows": test.tolist(),
    "profile_groups": groups.tolist(),
    "cv_indices_relative_to_train": [
        {"train": a.tolist(), "validation": b.tolist()} for a, b in cv
    ],
}
(output / "splits.json").write_text(json.dumps(split_record, indent=2) + "\n")
print(f"Train: {len(train)} rows; test: {len(test)} rows ({y_test.mean():.1%} churn).", flush=True)


## 3. Train and compare

## Train and compare

Our baseline gives every row the churn proportion observed in training. It uses
no test labels. AutoML compares Baseline, Linear, Decision Tree, and LightGBM,
and tries an ensemble. It selects by **validation log loss**, which penalizes
confident wrong probabilities. Lower is better.

Feature discovery is included in MLJAR's base installation, but is disabled here
because this example uses custom grouped validation. Other feature engineering
and stacking are disabled to keep this first example focused.


In [ ]:
# Step 3. Train a constant-probability baseline and compare AutoML models.
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
automl = AutoML(
    results_path=str(output / "automl"), mode="Explain",
    algorithms=["Baseline", "Linear", "Decision Tree", "LightGBM"],
    ml_task="binary_classification", eval_metric="logloss", total_time_limit=budget,
    validation_strategy={"validation_type": "custom"},
    feature_discovery=False, golden_features=False, kmeans_features=False,
    features_selection=False, stack_models=False, train_ensemble=True,
    explain_level=1, n_jobs=2, random_state=SEED,
)
fit_started = time.perf_counter()
automl.fit(X_train, y_train, cv=cv)
fit_seconds = time.perf_counter() - fit_started


## 4. Evaluate probabilities and errors

## Understand the predictions

`predict_proba()` returns probabilities for both classes. For this numeric 0/1
target, column 1 represents churn:

```python
probability_of_churn = automl.predict_proba(X_test)[:, 1]
```

A value of 0.7 is the model's estimated churn probability. It is not a guarantee
that exactly 70% of such customers will leave: calibration on representative
future data still needs checking. We turn probabilities into labels using a
**0.5 threshold fixed before evaluation**, rather than tuning against the test set.

- A **false positive** flags a customer who stayed, potentially wasting outreach.
- A **false negative** misses a customer who churned.
- **Precision** asks how many flagged customers actually churned.
- **Recall** asks how many customers who churned were flagged.

A real threshold depends on outreach capacity and error costs. Choose it on
separate validation data, then assess it once on untouched test data. This tutorial
uses its explicit threshold rather than `predict()`, which can use MLJAR's own
selected classification threshold.


In [ ]:
# Step 4. Evaluate probabilities and classification errors on untouched rows.
# For our numeric target 0/1, the second probability column is class 1 (churn).
probabilities = automl.predict_proba(X_test)[:, 1]
baseline_probabilities = baseline.predict_proba(X_test)[:, 1]
scores = {"baseline": evaluate(y_test, baseline_probabilities),
          "automl": evaluate(y_test, probabilities)}
print(pd.DataFrame(scores).drop(index="confusion_matrix").to_string(), flush=True)
X_test.assign(source_row=test, actual_churn=y_test,
              churn_probability=probabilities,
              predicted_churn=(probabilities >= THRESHOLD).astype(int),
              baseline_probability=baseline_probabilities).to_csv(output / "test_predictions.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout="constrained")
for ax, (name, result) in zip(axes, scores.items()):
    ConfusionMatrixDisplay(np.array(result["confusion_matrix"]),
                           display_labels=["Stayed", "Churned"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name.title()} · threshold {THRESHOLD}")
fig.savefig(output / "confusion-matrix.png", dpi=160)
plt.close(fig)
display(Image(filename=str(output / "confusion-matrix.png")))


The matrices compare the baseline and model on the same untouched rows.
Rows are actual outcomes; columns are predictions. Baseline precision is reported
as zero by convention because it flags no customers at 0.5. It is not an estimate
of the reliability of any flagged group. All scores are row-weighted.

The printed metrics are from **your run**. Compare them with the recorded tutorial
without treating that single split as a promise of future performance. Do not tune
models or thresholds against these test results.


## 5. Predict and reload

Keep the full `automl` directory. These example inputs are held-out rows without their labels. The next cell verifies all test probabilities after loading in a separate interpreter.


In [ ]:
# Step 5. Predict example inputs and verify reload in a fresh interpreter.
# These are held-out examples with the label removed, not newly collected customers.
new_rows = X_test.head(3)
new_rows.to_csv(output / "new_customers.csv", index=False)
print("Example churn probabilities:", automl.predict_proba(new_rows)[:, 1], flush=True)
X_test.to_csv(output / "reload_inputs.csv", index=False)
subprocess.run([
    sys.executable, "-c", textwrap.dedent("""
    import sys
    import numpy as np
    import pandas as pd
    from supervised import AutoML
    model = AutoML(results_path=sys.argv[1])
    inputs = pd.read_csv(sys.argv[2])
    np.save(sys.argv[3], model.predict_proba(inputs)[:, 1])
    """),
    str(output / "automl"), str(output / "reload_inputs.csv"),
    str(output / "reloaded_probabilities.npy"),
], check=True)
reloaded = np.load(output / "reloaded_probabilities.npy")
np.testing.assert_allclose(probabilities, reloaded, rtol=1e-7, atol=1e-7)
print("Fresh-process reload agrees with the original probabilities.", flush=True)


## 6. Read the report

The next cell saves versions and evaluation metrics and displays the report. Click a model name to inspect its validation metrics and settings. Validation scores use different rows from the test scores above. Importance describes model reliance, not causation.


In [ ]:
# Step 6. Create the HTML report and record this run.
report = automl.report()
metrics = {
    "dataset_url": DATA_URL, "dataset_sha256": DATA_SHA256,
    "features": FEATURES, "seed": SEED, "threshold": THRESHOLD,
    "train_rows": len(train), "test_rows": len(test),
    "train_churned": int(y_train.sum()), "test_churned": int(y_test.sum()),
    "profile_groups": int(groups.nunique()), "scores": scores,
    "training_budget_seconds": budget, "fit_wall_seconds": fit_seconds,
    "workflow_seconds_excluding_imports": time.perf_counter() - started,
    "reload_max_absolute_difference": float(np.max(np.abs(probabilities - reloaded))),
    "python": platform.python_version(), "platform": platform.platform(),
    "package_versions": {name: importlib.metadata.version(name) for name in (
        "mljar-supervised", "numpy", "pandas", "scikit-learn", "lightgbm", "features-goldmine", "joblib")},
}
(output / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
print(f"Open the report: {(output / 'automl' / 'README.html').as_uri()}", flush=True)
print("Keep the complete automl directory for future predictions.", flush=True)
thread_limits.restore_original_limits()
display(report)


## Predict another CSV and reload

The script saves `new_customers.csv`: three held-out example rows without labels,
not independently collected customers. In a new Python process:

```python
import pandas as pd
from supervised import AutoML

model = AutoML(results_path="AutoML_CustomerChurn_Notebook/automl")
customers = pd.read_csv("AutoML_CustomerChurn_Notebook/new_customers.csv")
print(model.predict_proba(customers)[:, 1])
```

Provide the same ten column names and units. Use tariff labels `Pay as you go`
and `Contractual`, as in the saved CSV. The complete script verifies all held-out
probabilities against a fresh-process reload with a small numerical tolerance.


## Adapt it to your own customers

Replace the download helper with your CSV reader, choose a binary target, and
keep only inputs available at the prediction point. Replace profile grouping
with real customer/account groups or a time split when those are available.
Set aside test data before training, compare against a baseline, and review
errors and probability quality. Use a new output path for every training run.

If a checksum check fails, inspect the source rather than deleting the check.
If the output directory exists, choose a new `--output` name or reload the saved
model. The `--data` option accepts a previously downloaded, verified source CSV.

For Studio, an intended starting prompt is: “Help me run this churn notebook,
explain the probabilities and errors, and show the report.” The Studio route and
an unassisted beginner walkthrough remain unvalidated. The Python and notebook
runs have their own recorded evidence below.


## Attribution

Iranian Churn [Dataset] (2020), UCI Machine Learning Repository. [DOI: 10.24432/C5JW3Z](https://doi.org/10.24432/C5JW3Z), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Snapshot retrieved 27 October 2026. The source download is unchanged; column spacing and tariff labels are normalized in memory. Full [tutorial and evidence](https://supervised.mljar.com/tutorials/customer-churn/).
